In [ ]:

#### Package Loading and Pathing ####
import tensorflow as tf
import numpy as np
import napari
import tifffile as tiff
from pathlib import Path
import platform
import nd2
import pandas as pd


SYSTEM = platform.system()

if SYSTEM == "Linux":
    DATA_ROOT = Path("/home/tdeibert/Data")
else:
    DATA_ROOT = Path(r"C:/Users/cowboy/OneDrive/Documents/Unviversity of Alabama/")

# General dataset roots
DATASETS_ROOT = DATA_ROOT / "Nuclear_Scaling" / "Data_Sets"

# ML pipeline roots
ML_ROOT   = DATA_ROOT / "Python_Coding" / "machine_learning" / "Data"
IMG_DIR   = ML_ROOT / "images"
LAB_DIR   = ML_ROOT / "labels"
MODEL_DIR = ML_ROOT / "models"
OUT_DIR   = ML_ROOT / "outputs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

assert IMG_DIR.exists(), f"Missing: {IMG_DIR}"
assert LAB_DIR.exists(), f"Missing: {LAB_DIR}"

print("System:", SYSTEM)
print("Data root:", DATA_ROOT)

# -----------------------------------
# Mask Paths 
# -----------------------------------
#### Mask Paths ####

NUCLEUS_MASK_NAME = "nucleus_mask.tif"
DROPLET_MASK_NAME = "droplet_mask.tif"

NUCLEUS_MASK_PATH = OUT_DIR / NUCLEUS_MASK_NAME
DROPLET_MASK_PATH = OUT_DIR / DROPLET_MASK_NAME

assert NUCLEUS_MASK_PATH.exists(), f"Missing nucleus mask: {NUCLEUS_MASK_PATH}"
assert DROPLET_MASK_PATH.exists(), f"Missing droplet mask: {DROPLET_MASK_PATH}"

print("Nucleus mask path:", NUCLEUS_MASK_PATH)
print("Droplet mask path:", DROPLET_MASK_PATH)

# -----------------------------------
# Experiment selection
# -----------------------------------

condition = "Control"
replicate = "Extract_3"
image_name = "Untitled.tif"

IMG_PATH = DATASETS_ROOT / condition / replicate / image_name

assert IMG_PATH.exists(), f"Image not found: {IMG_PATH}"

print("Condition:", condition)
print("Replicate:", replicate)
print("Image:", image_name)
print("Image path:", IMG_PATH)

#### Image Shape and Metadata Exploration ####

with tiff.TiffFile(IMG_PATH) as tif:

    series = tif.series[0]
    shape = series.shape
    axes = series.axes

    print("Number of pages:", len(tif.pages))
    print("Series shape:", shape)
    print("Axes:", axes)

    axis_sizes = dict(zip(axes, shape))

    n_time = axis_sizes.get("T", 1)
    n_z = axis_sizes.get("Z", 1)
    n_channel = axis_sizes.get("C", 1)
    n_y = axis_sizes.get("Y", 1)
    n_x = axis_sizes.get("X", 1)

print("n_time:", n_time)
print("n_z:", n_z)
print("n_channel:", n_channel)
print("n_y:", n_y)
print("n_x:", n_x)

#### Channel Definitions ####

NLS_CHANNEL = 0
MEMBRANE_CHANNEL = 1
NUCLEAR_PORE_CHANNEL = 2

CHANNEL_NAMES = [
    "NLS",
    "Membrane",
    "Nuclear_Pore"
]

assert n_channel == 3, f"Expected 3 channels, found {n_channel}"

#### Image Loading ####

MASTER_IMAGE = tiff.memmap(IMG_PATH)

print("Master image shape:", MASTER_IMAGE.shape)

In [ ]:
#### Image Loader Helpers#### 
def reorder_to_tzcyx(arr, axes):
    """
    Reorder an image array to TZCYX.

    Parameters
    ----------
    arr : ndarray
        Input array from tifffile.
    axes : str
        Axis string from tifffile, e.g. 'TZCYX', 'ZCYX', 'CYX', etc.

    Returns
    -------
    arr_tzcyx : ndarray
        Output array with shape (T, Z, C, Y, X)
    """
    axes = list(axes)

    # Add missing singleton axes
    target_axes = ["T", "Z", "C", "Y", "X"]

    for ax in target_axes:
        if ax not in axes:
            arr = np.expand_dims(arr, axis=0)
            axes = [ax] + axes

    # Now reorder
    perm = [axes.index(ax) for ax in target_axes]
    arr_tzcyx = np.transpose(arr, perm)

    return arr_tzcyx
# -----------------------------------
# Mask Loading Helpers
# -----------------------------------

def reorder_mask_to_tzyx(arr, axes):
    """
    Reorder a mask array to TZYX.
    Accepts axes like TZYX, ZYX, TYX, etc.
    """
    axes = list(axes)
    target_axes = ["T", "Z", "Y", "X"]

    for ax in target_axes:
        if ax not in axes:
            arr = np.expand_dims(arr, axis=0)
            axes = [ax] + axes

    perm = [axes.index(ax) for ax in target_axes]
    arr_tzyx = np.transpose(arr, perm)

    return arr_tzyx

In [ ]:
#-----------------------------------
# Image Loading and Mask Loading
#-----------------------------------

#Image
with tiff.TiffFile(image_path) as Main_Image:
    image = tiff.memmap(image_path)
    #print("Number of Time Points", Main_Image.series[1].shape)
    print("series shape:" , Main_Image.series[0].shape) 
    print("Images Axis", Main_Image.series[0].axes) 
    axes = Main_Image.series[0].axes
    shape = Main_Image.series[0].shape 
    axis_dict = dict(zip(axes, shape))
    c_axis_index = axes.index("C")
    nuclei_channel_index = 1 
    ome = Main_Image.ome_metadata

print("Number of Time Points:", axis_dict["T"])
print("Number of Z slices:", axis_dict["Z"])
print("Number of Channels:", axis_dict["C"])
print("Image size (Y, X):", axis_dict["Y"], axis_dict["X"])
print(ome)

#Masks 
with tiff.TiffFile(NUCLEUS_MASK_PATH) as tif:
    nucleus_series = tif.series[0]
    nucleus_axes = nucleus_series.axes

with tiff.TiffFile(DROPLET_MASK_PATH) as tif:
    droplet_series = tif.series[0]
    droplet_axes = droplet_series.axes

NUCLEUS_MASK_STACK = tiff.memmap(NUCLEUS_MASK_PATH)
DROPLET_MASK_STACK = tiff.memmap(DROPLET_MASK_PATH)

NUCLEUS_MASK_STACK = reorder_mask_to_tzyx(NUCLEUS_MASK_STACK, nucleus_axes)
DROPLET_MASK_STACK = reorder_mask_to_tzyx(DROPLET_MASK_STACK, droplet_axes)

print("Nucleus mask shape (T, Z, Y, X):", NUCLEUS_MASK_STACK.shape)
print("Droplet mask shape (T, Z, Y, X):", DROPLET_MASK_STACK.shape)

# Shape Checks 
assert MASTER_IMAGE.shape[0] == NUCLEUS_MASK_STACK.shape[0], "Time dimension mismatch: raw vs nucleus mask"
assert MASTER_IMAGE.shape[1] == NUCLEUS_MASK_STACK.shape[1], "Z dimension mismatch: raw vs nucleus mask"
assert MASTER_IMAGE.shape[3] == NUCLEUS_MASK_STACK.shape[2], "Y dimension mismatch: raw vs nucleus mask"
assert MASTER_IMAGE.shape[4] == NUCLEUS_MASK_STACK.shape[3], "X dimension mismatch: raw vs nucleus mask"

assert MASTER_IMAGE.shape[0] == DROPLET_MASK_STACK.shape[0], "Time dimension mismatch: raw vs droplet mask"
assert MASTER_IMAGE.shape[1] == DROPLET_MASK_STACK.shape[1], "Z dimension mismatch: raw vs droplet mask"
assert MASTER_IMAGE.shape[3] == DROPLET_MASK_STACK.shape[2], "Y dimension mismatch: raw vs droplet mask"
assert MASTER_IMAGE.shape[4] == DROPLET_MASK_STACK.shape[3], "X dimension mismatch: raw vs droplet mask"

In [ ]:
#### Image Visualization with Napari #### 
import napari
viewer = napari.Viewer()
viewer.add_image(
    image,
    name = "Control_Extract_3",
    channel_axis = 2
)

Start of Analysis section 

In [ ]:
"""
Integrated nucleus + droplet ROI analysis pipeline

What this script does
---------------------
For each time point (T) and Z-plane (Z):

1. Uses the nucleus mask stack to define nuclear ROIs
2. Uses the droplet mask stack to define droplet ROIs
3. Assigns each nucleus to its corresponding droplet
4. Builds overlap-safe, droplet-constrained nuclear halos
5. Measures nuclear area and intensity features on the raw image
6. Computes an N/C-style ratio using:
       (nucleus_boundary_mean - nucleus_inner_mean) /
       (outermost_halo_mean - nucleus_boundary_mean)
7. Runs a radial sweep / polar analysis from the nucleus centroid outward
   until the droplet boundary
8. Lets you choose:
      - which raw channel to use for halo/NC measurements
      - which raw channel to use for radial sweep measurements

Expected input shapes
---------------------
raw_stack            : (T, Z, C, Y, X)
nucleus_mask_stack   : (T, Z, Y, X)
droplet_mask_stack   : (T, Z, Y, X)

All geometry and measurements are done in PIXELS.
No micron scaling is performed inside the analysis.
Convert to microns later if needed.

Dependencies
------------
numpy
pandas
scipy
scikit-image
"""

from scipy.ndimage import gaussian_filter, distance_transform_edt
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, remove_small_holes


# =============================================================================
# USER SETTINGS
# =============================================================================

# Channel used for halo / nuclear boundary / N-C measurements
HALO_CHANNEL_INDEX = NLS_CHANNEL

# Channel used for radial sweep / polar transform measurements
RADIAL_CHANNEL_INDEX = MEMBRANE_CHANNEL

# Nucleus cleanup settings
NUCLEUS_SIGMA = 1.0
NUCLEUS_THRESHOLD = 0.5
NUCLEUS_MIN_AREA = 40
NUCLEUS_MAX_AREA = 5000
NUCLEUS_FILL_HOLES = True
NUCLEUS_HOLE_AREA_THRESHOLD = 25

# Droplet cleanup settings
DROPLET_SIGMA = 1.0
DROPLET_THRESHOLD = 0.5
DROPLET_MIN_AREA = 500
DROPLET_MAX_AREA = 1000000
DROPLET_FILL_HOLES = True
DROPLET_HOLE_AREA_THRESHOLD = 100

# ROI geometry settings (all in PIXELS)
BOUNDARY_WIDTH = 4   # boundary ROI INSIDE nucleus
HALO_WIDTH = 4       # halo width OUTSIDE nucleus
N_HALOS = 4

# Radial sweep settings
N_ANGLES = 180
RADIAL_STEP_SIZE = 1.0
EXCLUDE_NUCLEUS_FROM_RADIAL = True
MAX_RAY_STEPS = 4000


In [ ]:
# =============================================================================
# BASIC HELPERS
# =============================================================================

def safe_mean(values):
    return np.nan if values.size == 0 else float(np.mean(values))


def safe_sum(values):
    return np.nan if values.size == 0 else float(np.sum(values))


def safe_max(values):
    return np.nan if values.size == 0 else float(np.max(values))


def safe_std(values):
    return np.nan if values.size == 0 else float(np.std(values))


In [ ]:
# =============================================================================
# INPUT VALIDATION
# =============================================================================

def validate_inputs(raw_stack, nucleus_mask_stack, droplet_mask_stack,
                    halo_channel_index, radial_channel_index):
    if raw_stack.ndim != 5:
        raise ValueError("raw_stack must have shape (T, Z, C, Y, X)")

    if nucleus_mask_stack.ndim != 4:
        raise ValueError("nucleus_mask_stack must have shape (T, Z, Y, X)")

    if droplet_mask_stack.ndim != 4:
        raise ValueError("droplet_mask_stack must have shape (T, Z, Y, X)")

    if nucleus_mask_stack.shape != droplet_mask_stack.shape:
        raise ValueError("nucleus_mask_stack and droplet_mask_stack must have identical shape")

    t_raw, z_raw, c_raw, y_raw, x_raw = raw_stack.shape
    t_mask, z_mask, y_mask, x_mask = nucleus_mask_stack.shape

    if (t_raw != t_mask) or (z_raw != z_mask) or (y_raw != y_mask) or (x_raw != x_mask):
        raise ValueError(
            "raw_stack shape must match masks in T, Z, Y, X. "
            f"Got raw_stack {raw_stack.shape} and masks {nucleus_mask_stack.shape}"
        )

    if not (0 <= halo_channel_index < c_raw):
        raise ValueError(f"HALO_CHANNEL_INDEX={halo_channel_index} is out of bounds for C={c_raw}")

    if not (0 <= radial_channel_index < c_raw):
        raise ValueError(f"RADIAL_CHANNEL_INDEX={radial_channel_index} is out of bounds for C={c_raw}")


In [ ]:
# =============================================================================
# MASK CLEANING
# =============================================================================

def clean_binary_mask(mask_2d, sigma, threshold, min_area, max_area,
                      fill_holes=True, hole_area_threshold=25):
    """
    Clean a 2D binary mask.

    Notes:
    - Gaussian blur is applied only to the binary mask used for geometry cleanup.
    - Measurements are NOT taken from the blurred image.
    """
    mask_bin = mask_2d > 0

    blurred = gaussian_filter(mask_bin.astype(float), sigma=sigma)
    thresh = blurred > threshold

    thresh = remove_small_objects(thresh, min_size=min_area)

    if fill_holes:
        thresh = remove_small_holes(thresh, area_threshold=hole_area_threshold)

    labeled = label(thresh)
    cleaned = np.zeros_like(thresh, dtype=bool)

    for region in regionprops(labeled):
        if min_area <= region.area <= max_area:
            coords = region.coords
            cleaned[coords[:, 0], coords[:, 1]] = True

    labeled_clean = label(cleaned)
    return cleaned.astype(np.uint8), labeled_clean


In [ ]:
# =============================================================================
# NUCLEUS -> DROPLET ASSIGNMENT
# =============================================================================

def assign_nucleus_to_droplet(nucleus_mask, droplet_labeled_2d):
    """
    Assign one nucleus to the droplet label with the greatest overlap.
    Returns 0 if no overlap exists.
    """
    overlapping = droplet_labeled_2d[nucleus_mask]
    overlapping = overlapping[overlapping > 0]

    if overlapping.size == 0:
        return 0

    labels, counts = np.unique(overlapping, return_counts=True)
    return int(labels[np.argmax(counts)])


def build_nucleus_droplet_map(nucleus_labeled_2d, droplet_labeled_2d):
    """
    Returns dict: nucleus_id -> droplet_id
    """
    mapping = {}

    for region in regionprops(nucleus_labeled_2d):
        nucleus_id = region.label
        nucleus_mask = nucleus_labeled_2d == nucleus_id
        droplet_id = assign_nucleus_to_droplet(nucleus_mask, droplet_labeled_2d)
        mapping[nucleus_id] = droplet_id

    return mapping

In [2]:
# =============================================================================
# OVERLAP-SAFE NUCLEAR TERRITORY
# =============================================================================

def build_nearest_nucleus_label_map(nucleus_labeled_2d):
    """
    Assign each background pixel to the nearest nucleus label.

    Returns
    -------
    nearest_label_map : 2D int array
        For every pixel, the nearest nucleus label
    outside_dist : 2D float array
        Distance from each pixel to the nearest nucleus pixel
    """
    nucleus_mask = nucleus_labeled_2d > 0

    outside_dist, nearest_idx = distance_transform_edt(
        ~nucleus_mask,
        return_indices=True
    )

    rr = nearest_idx[0]
    cc = nearest_idx[1]

    nearest_label_map = np.zeros_like(nucleus_labeled_2d, dtype=np.int32)
    nearest_label_map[~nucleus_mask] = nucleus_labeled_2d[rr[~nucleus_mask], cc[~nucleus_mask]]
    nearest_label_map[nucleus_mask] = nucleus_labeled_2d[nucleus_mask]

    return nearest_label_map, outside_dist


In [ ]:
# =============================================================================
# REGION BUILDING
# =============================================================================

def build_nuclear_regions_overlap_safe(
    nucleus_labeled_2d,
    nucleus_id,
    droplet_id,
    droplet_labeled_2d,
    nearest_label_map,
    outside_dist,
    boundary_width=4,
    halo_width=4,
    n_halos=4
):
    """
    Build nuclear full ROI, inner ROI, boundary ROI, and overlap-safe,
    droplet-constrained halos for one nucleus.
    """
    nucleus_mask = nucleus_labeled_2d == nucleus_id

    if nucleus_mask.sum() == 0:
        raise ValueError(f"Nucleus {nucleus_id} not found")

    inside_dist = distance_transform_edt(nucleus_mask)

    nucleus_full = nucleus_mask.copy()
    nucleus_boundary = nucleus_mask & (inside_dist <= boundary_width)
    nucleus_inner = nucleus_mask & (inside_dist > boundary_width)

    region_masks = {
        "nucleus_full": nucleus_full,
        "nucleus_inner": nucleus_inner,
        "nucleus_boundary": nucleus_boundary,
    }

    # If no assigned droplet exists, halos are empty
    if droplet_id == 0:
        for i in range(1, n_halos + 1):
            region_masks[f"H{i}"] = np.zeros_like(nucleus_mask, dtype=bool)
        return region_masks

    droplet_mask = droplet_labeled_2d == droplet_id

    assigned_background = (
        (nucleus_labeled_2d == 0) &
        (nearest_label_map == nucleus_id) &
        droplet_mask
    )

    for i in range(1, n_halos + 1):
        inner = (i - 1) * halo_width
        outer = i * halo_width

        halo = assigned_background & (outside_dist > inner) & (outside_dist <= outer)
        region_masks[f"H{i}"] = halo

    return region_masks

In [ ]:
# =============================================================================
# NUCLEUS / HALO MEASUREMENTS
# =============================================================================

def compute_nc_ratio(boundary_mean, inner_mean, outer_mean):
    """
    Current N/C-style ratio:
        (boundary_mean - inner_mean) / (outer_mean - boundary_mean)
    """
    numerator = boundary_mean - inner_mean
    denominator = outer_mean - boundary_mean

    if np.isnan(numerator) or np.isnan(denominator) or np.isclose(denominator, 0):
        return np.nan

    return numerator / denominator


def measure_single_nucleus(
    raw_img_halo_2d,
    nucleus_labeled_2d,
    droplet_labeled_2d,
    nucleus_id,
    droplet_id,
    nearest_label_map,
    outside_dist,
    boundary_width=4,
    halo_width=4,
    n_halos=4
):
    region_masks = build_nuclear_regions_overlap_safe(
        nucleus_labeled_2d=nucleus_labeled_2d,
        nucleus_id=nucleus_id,
        droplet_id=droplet_id,
        droplet_labeled_2d=droplet_labeled_2d,
        nearest_label_map=nearest_label_map,
        outside_dist=outside_dist,
        boundary_width=boundary_width,
        halo_width=halo_width,
        n_halos=n_halos
    )

    measurements = {}

    measurements["nuclear_area_px"] = int(region_masks["nucleus_full"].sum())

    if droplet_id > 0:
        droplet_mask = droplet_labeled_2d == droplet_id
        measurements["droplet_area_px"] = int(droplet_mask.sum())
    else:
        measurements["droplet_area_px"] = 0

    for region_name, region_mask in region_masks.items():
        vals = raw_img_halo_2d[region_mask]
        measurements[f"{region_name}_n_pixels"] = int(region_mask.sum())
        measurements[f"{region_name}_mean"] = safe_mean(vals)
        measurements[f"{region_name}_sum"] = safe_sum(vals)
        measurements[f"{region_name}_max"] = safe_max(vals)
        measurements[f"{region_name}_std"] = safe_std(vals)

    boundary_mean = measurements["nucleus_boundary_mean"]
    inner_mean = measurements["nucleus_inner_mean"]
    outer_mean = measurements[f"H{n_halos}_mean"]

    measurements["nc_ratio"] = compute_nc_ratio(
        boundary_mean=boundary_mean,
        inner_mean=inner_mean,
        outer_mean=outer_mean
    )

    return measurements, region_masks


In [ ]:
# =============================================================================
# RADIAL SWEEP
# =============================================================================

def sample_ray_to_droplet_edge(
    image_2d,
    droplet_mask,
    nucleus_mask,
    center_y,
    center_x,
    theta,
    step_size=1.0,
    exclude_nucleus=True,
    max_steps=4000
):
    """
    Sample a single radial ray from nucleus centroid outward until leaving droplet.
    All distances are in pixels.
    """
    ys = []
    xs = []
    vals = []
    dists = []
    inside_nucleus_flags = []

    for step in range(max_steps):
        r = step * step_size
        y = center_y + r * np.sin(theta)
        x = center_x + r * np.cos(theta)

        yi = int(round(y))
        xi = int(round(x))

        if yi < 0 or yi >= image_2d.shape[0] or xi < 0 or xi >= image_2d.shape[1]:
            break

        if not droplet_mask[yi, xi]:
            break

        is_nucleus = bool(nucleus_mask[yi, xi])

        if exclude_nucleus and is_nucleus:
            continue

        ys.append(yi)
        xs.append(xi)
        vals.append(image_2d[yi, xi])
        dists.append(r)
        inside_nucleus_flags.append(is_nucleus)

    return {
        "y": np.asarray(ys, dtype=int),
        "x": np.asarray(xs, dtype=int),
        "values": np.asarray(vals, dtype=float),
        "distance_px": np.asarray(dists, dtype=float),
        "inside_nucleus": np.asarray(inside_nucleus_flags, dtype=bool),
    }


def radial_profiles_for_nucleus(
    raw_img_radial_2d,
    nucleus_mask,
    droplet_mask,
    nucleus_id,
    droplet_id,
    t,
    z,
    center_y,
    center_x,
    n_angles=180,
    step_size=1.0,
    exclude_nucleus=True,
    max_steps=4000
):
    """
    Returns a long-form DataFrame with one row per sampled ray point.
    """
    rows = []
    thetas = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)

    for theta_idx, theta in enumerate(thetas):
        ray = sample_ray_to_droplet_edge(
            image_2d=raw_img_radial_2d,
            droplet_mask=droplet_mask,
            nucleus_mask=nucleus_mask,
            center_y=center_y,
            center_x=center_x,
            theta=theta,
            step_size=step_size,
            exclude_nucleus=exclude_nucleus,
            max_steps=max_steps
        )

        if ray["distance_px"].size == 0:
            continue

        ray_length_px = float(ray["distance_px"][-1])

        if np.isclose(ray_length_px, 0):
            distance_norm = np.zeros_like(ray["distance_px"])
        else:
            distance_norm = ray["distance_px"] / ray_length_px

        for i in range(ray["distance_px"].size):
            rows.append({
                "t": t,
                "z": z,
                "nucleus_id": nucleus_id,
                "droplet_id": droplet_id,
                "theta_index": theta_idx,
                "theta_rad": float(theta),
                "y": int(ray["y"][i]),
                "x": int(ray["x"][i]),
                "distance_px": float(ray["distance_px"][i]),
                "distance_norm": float(distance_norm[i]),
                "intensity": float(ray["values"][i]),
                "ray_length_px": ray_length_px,
                "inside_nucleus": bool(ray["inside_nucleus"][i]),
            })

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.DataFrame(rows)


def summarize_radial_profiles(radial_profiles_df):
    """
    Build one summary row per nucleus from the radial profile table.
    """
    if radial_profiles_df.empty:
        return pd.DataFrame()

    summary_rows = []

    group_cols = ["t", "z", "nucleus_id", "droplet_id"]
    grouped = radial_profiles_df.groupby(group_cols, dropna=False)

    for keys, sub in grouped:
        t, z, nucleus_id, droplet_id = keys

        mean_intensity = safe_mean(sub["intensity"].to_numpy())
        max_intensity = safe_max(sub["intensity"].to_numpy())
        mean_ray_length = safe_mean(sub["ray_length_px"].drop_duplicates().to_numpy())

        if sub["intensity"].notna().any():
            peak_idx = sub["intensity"].idxmax()
            peak_row = sub.loc[peak_idx]
            peak_distance_px = float(peak_row["distance_px"])
            peak_distance_norm = float(peak_row["distance_norm"])
        else:
            peak_distance_px = np.nan
            peak_distance_norm = np.nan

        summary_rows.append({
            "t": t,
            "z": z,
            "nucleus_id": nucleus_id,
            "droplet_id": droplet_id,
            "radial_mean_intensity": mean_intensity,
            "radial_max_intensity": max_intensity,
            "radial_mean_ray_length_px": mean_ray_length,
            "radial_peak_distance_px": peak_distance_px,
            "radial_peak_distance_norm": peak_distance_norm,
        })

    return pd.DataFrame(summary_rows)


In [ ]:
# =============================================================================
# ONE SLICE ANALYSIS
# =============================================================================

def analyze_single_tz_plane(
    raw_img_halo_2d,
    raw_img_radial_2d,
    nucleus_mask_2d,
    droplet_mask_2d,
    t,
    z,
    nucleus_sigma=1.0,
    nucleus_threshold=0.5,
    nucleus_min_area=40,
    nucleus_max_area=5000,
    nucleus_fill_holes=True,
    nucleus_hole_area_threshold=25,
    droplet_sigma=1.0,
    droplet_threshold=0.5,
    droplet_min_area=500,
    droplet_max_area=1000000,
    droplet_fill_holes=True,
    droplet_hole_area_threshold=100,
    boundary_width=4,
    halo_width=4,
    n_halos=4,
    n_angles=180,
    radial_step_size=1.0,
    exclude_nucleus_from_radial=True,
    max_ray_steps=4000
):
    """
    Analyze one T,Z plane.

    Returns
    -------
    nucleus_cleaned_2d
    nucleus_labeled_2d
    droplet_cleaned_2d
    droplet_labeled_2d
    object_measurements_df
    radial_profiles_df
    radial_summary_df
    """
    nucleus_cleaned_2d, nucleus_labeled_2d = clean_binary_mask(
        mask_2d=nucleus_mask_2d,
        sigma=nucleus_sigma,
        threshold=nucleus_threshold,
        min_area=nucleus_min_area,
        max_area=nucleus_max_area,
        fill_holes=nucleus_fill_holes,
        hole_area_threshold=nucleus_hole_area_threshold
    )

    droplet_cleaned_2d, droplet_labeled_2d = clean_binary_mask(
        mask_2d=droplet_mask_2d,
        sigma=droplet_sigma,
        threshold=droplet_threshold,
        min_area=droplet_min_area,
        max_area=droplet_max_area,
        fill_holes=droplet_fill_holes,
        hole_area_threshold=droplet_hole_area_threshold
    )

    if np.max(nucleus_labeled_2d) == 0:
        empty = pd.DataFrame()
        return (
            nucleus_cleaned_2d,
            nucleus_labeled_2d,
            droplet_cleaned_2d,
            droplet_labeled_2d,
            empty,
            empty,
            empty
        )

    nearest_label_map, outside_dist = build_nearest_nucleus_label_map(nucleus_labeled_2d)
    nucleus_to_droplet = build_nucleus_droplet_map(nucleus_labeled_2d, droplet_labeled_2d)

    object_rows = []
    radial_dfs = []

    for region in regionprops(nucleus_labeled_2d):
        nucleus_id = region.label
        droplet_id = nucleus_to_droplet.get(nucleus_id, 0)

        measurements, region_masks = measure_single_nucleus(
            raw_img_halo_2d=raw_img_halo_2d,
            nucleus_labeled_2d=nucleus_labeled_2d,
            droplet_labeled_2d=droplet_labeled_2d,
            nucleus_id=nucleus_id,
            droplet_id=droplet_id,
            nearest_label_map=nearest_label_map,
            outside_dist=outside_dist,
            boundary_width=boundary_width,
            halo_width=halo_width,
            n_halos=n_halos
        )

        row = {
            "t": t,
            "z": z,
            "nucleus_id": nucleus_id,
            "droplet_id": droplet_id,
            "centroid_y": float(region.centroid[0]),
            "centroid_x": float(region.centroid[1]),
            "bbox_min_row": int(region.bbox[0]),
            "bbox_min_col": int(region.bbox[1]),
            "bbox_max_row": int(region.bbox[2]),
            "bbox_max_col": int(region.bbox[3]),
        }
        row.update(measurements)
        object_rows.append(row)

        # Radial sweep only if assigned droplet exists
        if droplet_id > 0:
            nucleus_mask = nucleus_labeled_2d == nucleus_id
            droplet_mask = droplet_labeled_2d == droplet_id

            radial_df = radial_profiles_for_nucleus(
                raw_img_radial_2d=raw_img_radial_2d,
                nucleus_mask=nucleus_mask,
                droplet_mask=droplet_mask,
                nucleus_id=nucleus_id,
                droplet_id=droplet_id,
                t=t,
                z=z,
                center_y=float(region.centroid[0]),
                center_x=float(region.centroid[1]),
                n_angles=n_angles,
                step_size=radial_step_size,
                exclude_nucleus=exclude_nucleus_from_radial,
                max_steps=max_ray_steps
            )

            if not radial_df.empty:
                radial_dfs.append(radial_df)

    object_measurements_df = pd.DataFrame(object_rows)

    if len(radial_dfs) > 0:
        radial_profiles_df = pd.concat(radial_dfs, ignore_index=True)
    else:
        radial_profiles_df = pd.DataFrame()

    radial_summary_df = summarize_radial_profiles(radial_profiles_df)

    return (
        nucleus_cleaned_2d,
        nucleus_labeled_2d,
        droplet_cleaned_2d,
        droplet_labeled_2d,
        object_measurements_df,
        radial_profiles_df,
        radial_summary_df
    )


In [ ]:
# =============================================================================
# FULL STACK ANALYSIS
# =============================================================================

def run_integrated_analysis(
    raw_stack,
    nucleus_mask_stack,
    droplet_mask_stack,
    halo_channel_index=0,
    radial_channel_index=0,
    nucleus_sigma=1.0,
    nucleus_threshold=0.5,
    nucleus_min_area=40,
    nucleus_max_area=5000,
    nucleus_fill_holes=True,
    nucleus_hole_area_threshold=25,
    droplet_sigma=1.0,
    droplet_threshold=0.5,
    droplet_min_area=500,
    droplet_max_area=1000000,
    droplet_fill_holes=True,
    droplet_hole_area_threshold=100,
    boundary_width=4,
    halo_width=4,
    n_halos=4,
    n_angles=180,
    radial_step_size=1.0,
    exclude_nucleus_from_radial=True,
    max_ray_steps=4000
):
    validate_inputs(
        raw_stack=raw_stack,
        nucleus_mask_stack=nucleus_mask_stack,
        droplet_mask_stack=droplet_mask_stack,
        halo_channel_index=halo_channel_index,
        radial_channel_index=radial_channel_index
    )

    T, Z, C, Y, X = raw_stack.shape

    nucleus_cleaned_stack = np.zeros((T, Z, Y, X), dtype=np.uint8)
    nucleus_labeled_stack = np.zeros((T, Z, Y, X), dtype=np.int32)

    droplet_cleaned_stack = np.zeros((T, Z, Y, X), dtype=np.uint8)
    droplet_labeled_stack = np.zeros((T, Z, Y, X), dtype=np.int32)

    object_dfs = []
    radial_profile_dfs = []
    radial_summary_dfs = []

    for t in range(T):
        for z in range(Z):
            raw_img_halo_2d = raw_stack[t, z, halo_channel_index]
            raw_img_radial_2d = raw_stack[t, z, radial_channel_index]

            nucleus_mask_2d = nucleus_mask_stack[t, z]
            droplet_mask_2d = droplet_mask_stack[t, z]

            (
                nucleus_cleaned_2d,
                nucleus_labeled_2d,
                droplet_cleaned_2d,
                droplet_labeled_2d,
                object_measurements_df,
                radial_profiles_df,
                radial_summary_df
            ) = analyze_single_tz_plane(
                raw_img_halo_2d=raw_img_halo_2d,
                raw_img_radial_2d=raw_img_radial_2d,
                nucleus_mask_2d=nucleus_mask_2d,
                droplet_mask_2d=droplet_mask_2d,
                t=t,
                z=z,
                nucleus_sigma=nucleus_sigma,
                nucleus_threshold=nucleus_threshold,
                nucleus_min_area=nucleus_min_area,
                nucleus_max_area=nucleus_max_area,
                nucleus_fill_holes=nucleus_fill_holes,
                nucleus_hole_area_threshold=nucleus_hole_area_threshold,
                droplet_sigma=droplet_sigma,
                droplet_threshold=droplet_threshold,
                droplet_min_area=droplet_min_area,
                droplet_max_area=droplet_max_area,
                droplet_fill_holes=droplet_fill_holes,
                droplet_hole_area_threshold=droplet_hole_area_threshold,
                boundary_width=boundary_width,
                halo_width=halo_width,
                n_halos=n_halos,
                n_angles=n_angles,
                radial_step_size=radial_step_size,
                exclude_nucleus_from_radial=exclude_nucleus_from_radial,
                max_ray_steps=max_ray_steps
            )

            nucleus_cleaned_stack[t, z] = nucleus_cleaned_2d
            nucleus_labeled_stack[t, z] = nucleus_labeled_2d
            droplet_cleaned_stack[t, z] = droplet_cleaned_2d
            droplet_labeled_stack[t, z] = droplet_labeled_2d

            if not object_measurements_df.empty:
                object_dfs.append(object_measurements_df)

            if not radial_profiles_df.empty:
                radial_profile_dfs.append(radial_profiles_df)

            if not radial_summary_df.empty:
                radial_summary_dfs.append(radial_summary_df)

    object_measurements_df = (
        pd.concat(object_dfs, ignore_index=True)
        if len(object_dfs) > 0 else pd.DataFrame()
    )

    radial_profiles_df = (
        pd.concat(radial_profile_dfs, ignore_index=True)
        if len(radial_profile_dfs) > 0 else pd.DataFrame()
    )

    radial_summary_df = (
        pd.concat(radial_summary_dfs, ignore_index=True)
        if len(radial_summary_dfs) > 0 else pd.DataFrame()
    )

    return {
        "nucleus_cleaned_stack": nucleus_cleaned_stack,
        "nucleus_labeled_stack": nucleus_labeled_stack,
        "droplet_cleaned_stack": droplet_cleaned_stack,
        "droplet_labeled_stack": droplet_labeled_stack,
        "object_measurements_df": object_measurements_df,
        "radial_profiles_df": radial_profiles_df,
        "radial_summary_df": radial_summary_df,
    }

In [ ]:
# =============================================================================
# OPTIONAL POSTHOC MICRON CONVERSION
# =============================================================================

def add_area_um2_posthoc(df, pixel_size_um, area_column="nuclear_area_px", new_column="nuclear_area_um2"):
    df = df.copy()
    df[new_column] = df[area_column] * (pixel_size_um ** 2)
    return df


In [ ]:
results = run_integrated_analysis(
    raw_stack=MASTER_IMAGE,
    nucleus_mask_stack=NUCLEUS_MASK_STACK,
    droplet_mask_stack=DROPLET_MASK_STACK,
    halo_channel_index=HALO_CHANNEL_INDEX,
    radial_channel_index=RADIAL_CHANNEL_INDEX
)

In [ ]:
# =============================================================================
# EXAMPLE USAGE
# =============================================================================

if __name__ == "__main__":
    # -------------------------------------------------------------------------
    # Replace these example arrays with your real data
    # raw_stack shape must be (T, Z, C, Y, X)
    # nucleus_mask_stack and droplet_mask_stack shape must be (T, Z, Y, X)
    # -------------------------------------------------------------------------
    T, Z, C, Y, X = 2, 3, 2, 128, 128

    raw_stack = np.random.random((T, Z, C, Y, X)).astype(np.float32)

    # Example fake masks
    nucleus_mask_stack = (raw_stack[:, :, 0] > 0.96).astype(np.uint8)
    droplet_mask_stack = (raw_stack[:, :, 1] > 0.80).astype(np.uint8)

    results = run_integrated_analysis(
        raw_stack=raw_stack,
        nucleus_mask_stack=nucleus_mask_stack,
        droplet_mask_stack=droplet_mask_stack,
        halo_channel_index=HALO_CHANNEL_INDEX,
        radial_channel_index=RADIAL_CHANNEL_INDEX,
        nucleus_sigma=NUCLEUS_SIGMA,
        nucleus_threshold=NUCLEUS_THRESHOLD,
        nucleus_min_area=NUCLEUS_MIN_AREA,
        nucleus_max_area=NUCLEUS_MAX_AREA,
        nucleus_fill_holes=NUCLEUS_FILL_HOLES,
        nucleus_hole_area_threshold=NUCLEUS_HOLE_AREA_THRESHOLD,
        droplet_sigma=DROPLET_SIGMA,
        droplet_threshold=DROPLET_THRESHOLD,
        droplet_min_area=DROPLET_MIN_AREA,
        droplet_max_area=DROPLET_MAX_AREA,
        droplet_fill_holes=DROPLET_FILL_HOLES,
        droplet_hole_area_threshold=DROPLET_HOLE_AREA_THRESHOLD,
        boundary_width=BOUNDARY_WIDTH,
        halo_width=HALO_WIDTH,
        n_halos=N_HALOS,
        n_angles=N_ANGLES,
        radial_step_size=RADIAL_STEP_SIZE,
        exclude_nucleus_from_radial=EXCLUDE_NUCLEUS_FROM_RADIAL,
        max_ray_steps=MAX_RAY_STEPS
    )

    print("\nObject measurements:")
    print(results["object_measurements_df"].head())

    print("\nRadial profiles:")
    print(results["radial_profiles_df"].head())

    print("\nRadial summary:")
    print(results["radial_summary_df"].head())

    # Example posthoc conversion:
    # obj_df_um = add_area_um2_posthoc(results["object_measurements_df"], pixel_size_um=0.108)

In [ ]:
viewer = napari.Viewer()

viewer.add_image(
    MASTER_IMAGE,
    name="Raw_Image",
    channel_axis=2,
    axis_labels=["T", "Z", "C", "Y", "X"]
)

viewer.add_image(
    NUCLEUS_MASK_STACK,
    name="Nucleus_Mask"
)

viewer.add_image(
    DROPLET_MASK_STACK,
    name="Droplet_Mask"
)

napari.run()